# Safety Stock and Reorder Point

This notebook turns the demand stats from the forecast notebook into two numbers every planner needs: how much safety stock to hold, and at what stock level to place a new order.

In [1]:
import pandas as pd
import numpy as np
from scipy.stats import norm

sku_stats = pd.read_csv("../data/raw/sku_stats.csv")
sku_stats.head()

,sku_id,avg_daily_demand,demand_std
0,SKU-001,26.386010,9.446979
1,SKU-002,38.082902,7.734783
2,SKU-003,24.652850,8.494380
3,SKU-004,36.860104,9.359827
4,SKU-005,35.269430,10.115692


## Set the Planning Assumptions

Lead time is how many days it takes for a new order to arrive. Service level is how often we want to avoid running out of stock.

In [2]:
lead_time_days = 7
service_level = 0.95

z_score = norm.ppf(service_level)
print("z score for", service_level, "service level:", round(z_score, 2))

z score for 0.95 service level: 1.64


## Define the Formulas

Safety stock grows with demand variability and lead time. Reorder point is the expected demand during lead time plus the safety stock buffer.

In [3]:
def calculate_safety_stock(demand_std, lead_time_days, z_score):
    return z_score * demand_std * np.sqrt(lead_time_days)

def calculate_reorder_point(avg_daily_demand, lead_time_days, safety_stock):
    return avg_daily_demand * lead_time_days + safety_stock

## Apply the Formulas to Every SKU

In [4]:
sku_stats["safety_stock"] = sku_stats["demand_std"].apply(
    lambda std: calculate_safety_stock(std, lead_time_days, z_score)
)

sku_stats["reorder_point"] = sku_stats.apply(
    lambda row: calculate_reorder_point(row["avg_daily_demand"], lead_time_days, row["safety_stock"]),
    axis=1,
)

sku_stats.round(1)

,sku_id,avg_daily_demand,demand_std,safety_stock,reorder_point
0,SKU-001,26.4,9.4,41.1,225.8
1,SKU-002,38.1,7.7,33.7,300.2
2,SKU-003,24.7,8.5,37.0,209.5
3,SKU-004,36.9,9.4,40.7,298.8
4,SKU-005,35.3,10.1,44.0,290.9
5,SKU-006,43.3,7.4,32.3,335.3
6,SKU-007,32.1,4.3,18.8,243.5
7,SKU-008,39.7,5.6,24.2,302.2
8,SKU-009,70.8,6.9,30.1,525.6
9,SKU-010,49.4,7.0,30.5,376.1


## Save the Inventory Plan

The dashboard reads this file to show the final recommendation per SKU.

In [5]:
sku_stats.to_csv("../data/raw/inventory_plan.csv", index=False)
print("saved inventory_plan.csv")

saved inventory_plan.csv


## What This Means in Practice

A SKU with high demand variability, like the Apparel SKUs, ends up with a much bigger safety stock than a stable SKU with the same average demand. This is the trade off planners deal with every day: more safety stock means fewer stockouts, but it also means more holding cost.